# Docker for Data Engineers

**Goal:** Build a small containerized data system step by step.

Run Docker commands in **PowerShell or a terminal**, not inside notebook cells.

Move into the companion project folder:

```powershell
cd day_2/day_2_3_Docker/docker_data_engineering_demo
```

## Learning Path

```text
0. Quick recap: image, container, Dockerfile, registry, mount, port, Compose
1. Sanity check: docker run hello-world
2. Build a pipeline image that downloads and prepares data
3. Keep generated data with a bind mount
4. Look inside a running container with docker exec
5. Build and run the FastAPI image
6. Write and run the dashboard image
7. Connect API and dashboard with Docker Compose
8. Discuss when a scheduler such as Airflow becomes useful
```


## 1. Quick Recap

Start with the working definitions. You will use each concept later in the commands.

| Concept | Working definition |
|---|---|
| Image | A packaged application environment, like a reusable template |
| Container | A running or stopped instance of an image |
| Dockerfile | The build instructions for an image |
| Registry | A place to download or publish images, such as Docker Hub |
| Bind mount | A host folder made visible inside a container |
| Port mapping | A host port forwarded to a container port |
| Docker Compose | A YAML file plus commands for running related services together |

Today the main idea is not memorizing terms. We will meet each term again while building something.


## 2. Address Cheat Sheet

This is the table worth keeping nearby. Most Docker confusion comes from forgetting **who is making the request**.

| Request comes from | Target | Use this address |
|---|---|---|
| Browser on your laptop | Published API container | `http://localhost:8000` |
| Browser on your laptop | Published dashboard container | `http://localhost:8501` |
| Container | Service running directly on your laptop | `http://host.docker.internal:<port>` |
| Container | Another service in the same Compose project | `http://<service_name>:<container_port>` |
| Process inside one container | Another process in the same container | `http://localhost:<port>` |

Mental model:

```text
localhost / 127.0.0.1 always means myself.

Browser's myself              = your laptop
API container's myself        = API container
Dashboard container's myself  = dashboard container
```

`0.0.0.0` is different: it is a server bind address. Inside a container, a web server usually listens on `0.0.0.0` so Docker can forward traffic into it.


## 3. First Docker Commands

Run in **PowerShell or a terminal**:

```powershell
docker --version
docker run hello-world
docker ps
docker ps -a
```

### What Docker Desktop Adds

Docker commands run in the terminal, but Docker Desktop is the local Docker environment that keeps the Docker Engine running on Windows and macOS. In class, use it as the control panel:

| Docker Desktop area | What students should notice |
|---|---|
| Containers | Which containers are running, stopped, or failed. This is the GUI version of `docker ps` and `docker ps -a`. |
| Images | Downloaded and locally built images, including their sizes and tags. |
| Logs / Inspect | Quick way to see why a container exited before jumping into terminal debugging. |
| Settings | Resource limits, WSL integration on Windows, file sharing, and whether Docker starts with the laptop. |

Before the hands-on part, ask students to check four things in Docker Desktop:

1. Docker Desktop says **Running**.
2. On Windows, WSL integration is enabled for the distro they use.
3. The project drive or folder is available to Docker, especially if bind mounts fail later.
4. Docker has enough memory for multiple containers; 4 GB is a comfortable classroom baseline.

Teaching note: Docker Desktop is not the container itself. It is the app that starts and manages the local Docker Engine. The image, container, network, and volume concepts are still the same concepts we use from the command line.

`hello-world` checks that Docker Desktop can start a container and pull an image from a registry.

`docker ps` shows running containers. `docker ps -a` also shows stopped containers.

To stop and remove a named container later:

```powershell
docker stop <container_name_or_id>
docker rm <container_name_or_id>
```


## 4. Build a Pipeline Image

Start with a useful data engineering job:

```text
taxi availability API -> raw JSON -> clean CSV -> DuckDB
rainfall API -> raw JSON -> clean CSV -> DuckDB
```

The companion project reuses the Day 1 / Day 2 data logic in `scripts/`. Docker is not changing the business logic. Docker is packaging the runtime needed to run it.

Open `Dockerfile.pipeline`:

```dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY requirements.pipeline.txt .
RUN pip install --no-cache-dir -r requirements.pipeline.txt

COPY scripts ./scripts
COPY reference ./reference

RUN mkdir -p /app/data

CMD ["sh", "-c", "python -m scripts.taxi_job && python -m scripts.rainfall_job"]
```

Read it line by line.

Important path idea: `/app` is a folder **inside the Docker image and container**. It is not your laptop's project folder. When Docker sees `WORKDIR /app`, it switches to `/app`; if `/app` does not exist yet, Docker creates it.

The build command later uses this folder as the build context:

```powershell
docker build -f Dockerfile.pipeline -t de-pipeline .
```

The final `.` means: send the current folder, `docker_data_engineering_demo/`, to Docker as the files available for `COPY`.

| Line | What it means |
|---|---|
| `FROM python:3.11-slim` | Start from an existing image that already has Python 3.11 installed. |
| `WORKDIR /app` | Use `/app` as the working folder inside the image/container. Docker creates it if needed. |
| `COPY requirements.pipeline.txt .` | Copy `requirements.pipeline.txt` from the current project folder on your laptop into the current container folder, `/app`. The `.` on the right means `/app` because of `WORKDIR /app`. |
| `RUN pip install --no-cache-dir -r requirements.pipeline.txt` | Install Python packages inside the image during the build. This runs in `/app`, so it can find `/app/requirements.pipeline.txt`. |
| `COPY scripts ./scripts` | Copy the local `scripts/` folder from your laptop into `/app/scripts` inside the image. |
| `COPY reference ./reference` | Copy the planning-area GeoJSON into `/app/reference` so the rainfall job can run even without a bind mount. |
| `RUN mkdir -p /app/data` | Create the default output folder inside the image. A bind mount can still replace it at runtime. |
| `CMD ["sh", "-c", "python -m scripts.taxi_job && python -m scripts.rainfall_job"]` | Define the default command when a container starts from this image. It runs taxi first, then rainfall. |

After the image is built, the code and dependencies are packaged inside the image. Running a container starts from that packaged environment.


### Build and Run the Pipeline Image

Run in **PowerShell or a terminal**:

```powershell
docker build -f Dockerfile.pipeline -t de-pipeline .
docker run --rm de-pipeline
```

What to notice:

| Option | Meaning |
|---|---|
| `-f Dockerfile.pipeline` | Use this Dockerfile instead of the default name |
| `-t de-pipeline` | Tag the image with a friendly name |
| `.` | Use the current folder as the build context. Docker can only `COPY` files from this folder and its subfolders during the build. |
| `--rm` | Remove the container after the command exits |

This pipeline is a **short-running container**. It starts, runs one job, writes output, and exits.

| Container type | Example in this lesson | What happens |
|---|---|---|
| Short-running job | Pipeline container | Runs one task and exits when the task is done. |
| Long-running service | API or dashboard container | Keeps running and waits for HTTP requests. |

The job runs, but any generated files are inside the container filesystem. Because `--rm` removes the container, those files are gone after the run.

The rainfall job also needs the planning-area GeoJSON. This image includes a default copy at `/app/reference/planning-areas.geojson`, so the simple `docker run --rm de-pipeline` command works. Later, Compose will mount the reference file explicitly to show the runtime version of the same idea.

That is a perfect setup for the next concept: bind mounts.


## 5. Keep Data with a Bind Mount

A container filesystem is temporary. Data engineers usually want generated JSON, CSV, Parquet, DuckDB files, logs, and model artifacts to remain available on the host machine.

Run the same image again, but mount the local `data/` folder into `/app/data` inside the container:

```powershell
docker run --rm -v ${PWD}/data:/app/data de-pipeline
```

After this command finishes, check two places:

| Where to look | What changed |
|---|---|
| Docker Desktop -> Images | The `de-pipeline` image is still there. Images remain after containers exit. |
| Local `data/` folder | New raw JSON, clean CSV, and DuckDB files were written through the bind mount. |

You may not see a running container in Docker Desktop because this command uses `--rm`. The container starts, runs the pipeline, exits, and is removed. The important visible result is the changed `data/` folder on your laptop.

Read the mount mapping carefully:

```text
-v ${PWD}/data:/app/data
   |          |
   |          `-- container path: where the folder appears inside the container
   `------------- host path: the real folder on your laptop
```

The left side and right side are different machines from Docker's point of view:

| Side | Example | Meaning |
|---|---|---|
| Host side | `${PWD}/data` | The folder on your laptop. This is where files remain after the container exits. |
| Container side | `/app/data` | The path that the program inside the container reads from and writes to. |

After the command finishes, inspect the local `data/` folder. It should contain files such as:

```text
data/raw/taxi_raw_<timestamp>.json
data/raw/rainfall_raw_<timestamp>.json
data/clean/taxi_clean_<timestamp>.csv
data/clean/rainfall_clean_<timestamp>.csv
data/basic_ingestion.duckdb
```

Why this matters:

| Without `-v` | With `-v ${PWD}/data:/app/data` |
|---|---|
| Output is written only inside the container filesystem. | Output is written to a host folder through the mount. |
| If the container is removed, the output is hard to recover. | Files stay on your laptop after the container exits. |
| Good for temporary computation. | Good for pipeline outputs, logs, databases, and artifacts. |

Important distinction:

| Thing | Where it belongs |
|---|---|
| `RUN mkdir -p /app/data` | Dockerfile, because it prepares a folder inside the image filesystem. |
| `-v ${PWD}/data:/app/data` | `docker run`, because it depends on the host machine and the runtime setup. |

Do not put a laptop path such as `${PWD}/data` inside a Dockerfile. A Dockerfile describes the image; the mount describes how a specific container run connects to your laptop.

> On Windows PowerShell, `${PWD}` usually works. If mounting fails, use an absolute host path.


## 6. Look Inside a Running Container with `docker exec`

A bind mount is easier to understand when you look inside the container.

The pipeline image normally creates short-running containers. That is good for batch jobs, but inconvenient for inspection because the container exits immediately after the job finishes.

To inspect it, start a temporary **long-running** container from the same image.

The pipeline image has this default command in its Dockerfile:

```dockerfile
CMD ["sh", "-c", "python -m scripts.taxi_job && python -m scripts.rainfall_job"]
```

That command runs the taxi and rainfall jobs and then exits. For inspection, we want the opposite: start the same container environment, mount the same data folder, but keep the container alive while we look around.

That is why the command below ends with `sleep infinity`:

```powershell
docker run -d --name pipeline-inspect -v ${PWD}/data:/app/data de-pipeline sleep infinity
```

Read the end of the command like this:

| Part | Meaning |
|---|---|
| `de-pipeline` | Start from the same image we built earlier. |
| `sleep` | The command Docker runs instead of the image's default pipeline command. |
| `infinity` | The argument passed to `sleep`. It tells `sleep` to wait forever, so the container stays running. |

`sleep infinity` is not doing data work. It is just a harmless command that keeps the container process alive. Once the container is alive, `docker exec` can enter it and run extra commands such as `sh`, `pwd`, and `ls`.

Check that it is running:

```powershell
docker ps
```

Also open **Docker Desktop -> Containers**. You should see a running container named `pipeline-inspect`. This is a good moment to connect the CLI and the GUI:

| CLI view | Docker Desktop view |
|---|---|
| `docker ps` | `pipeline-inspect` appears as running. |
| `docker logs pipeline-inspect` | The Logs tab may be quiet because `sleep infinity` does not print output. |
| `docker rm -f pipeline-inspect` | The container disappears or moves to stopped/removed state. |

This container is intentionally boring: it is only staying alive so we can inspect the filesystem and the mounted `data/` folder.

Now open a shell inside the running container:

```powershell
docker exec -it pipeline-inspect sh
```

Inside the container, run:

```sh
pwd
ls
ls /app
ls /app/data
ls /app/data/clean
```

What to notice:

| Command | What it shows |
|---|---|
| `pwd` | You are inside the container, usually in `/app`. |
| `ls /app` | The application files copied into the image. |
| `ls /app/data` | The mounted data folder. These files are actually stored on your laptop. |
| `ls /app/data/clean` | The clean CSV files generated by the pipeline. |

Exit the shell:

```sh
exit
```

Then remove the temporary inspection container:

```powershell
docker rm -f pipeline-inspect
```

`docker exec` does not create a new image and does not start a new container. It runs an extra command inside an already running container. That is why we first need a long-running container such as `pipeline-inspect`.

Short-running and long-running containers are both normal Docker patterns:

| Pattern | Typical command | Use case |
|---|---|---|
| Short-running | `docker run --rm de-pipeline` | Batch jobs, scripts, data ingestion, tests. |
| Long-running | `docker run -d ... de-api` | APIs, dashboards, databases, schedulers. |
| Temporary inspection | `docker run -d ... sleep infinity` + `docker exec -it ... sh` | Keep a container alive so you can look inside it. |

A few runtime options used in this lesson:

| Option | Example | Purpose |
|---|---|---|
| `-v` | `-v ${PWD}/data:/app/data` | Mount a host folder into a container. |
| `-p` | `-p 8000:8000` | Publish a container port on the host. |
| `-e` | `-e TAXI_API_URL=http://host.docker.internal:8000/taxi-latest` | Pass an environment variable into the container. |
| `-d` | `-d` | Run the container in the background. |
| `--name` | `--name pipeline-inspect` | Give the container a readable name. |
| command override | `de-pipeline sleep infinity` | Replace the image's default `CMD` for this one container run. |

We will use `-e` later for the dashboard. For now, just remember that it is a way to pass configuration into a container at runtime.


## 7. Build the FastAPI Image

The API container reads the DuckDB file generated by the pipeline container. Unlike the pipeline, the API is a **long-running service**: it keeps running and waits for requests.

```text
host data/basic_ingestion.duckdb -> mounted into API container -> FastAPI endpoints
```

Open `Dockerfile.api`:

```dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY requirements.api.txt .
RUN pip install --no-cache-dir -r requirements.api.txt

COPY api ./api

EXPOSE 8000

CMD ["uvicorn", "api.serving_api:app", "--host", "0.0.0.0", "--port", "8000"]
```

The `CMD` line is the container version of a normal command-line command. If you were running the API directly in PowerShell from the project folder, the command would look like this:

```powershell
python -m uvicorn api.serving_api:app --host 0.0.0.0 --port 8000
```

In the Dockerfile, we write the same idea in JSON-array form:

```dockerfile
CMD ["uvicorn", "api.serving_api:app", "--host", "0.0.0.0", "--port", "8000"]
```

Read the uvicorn command:

| Part | Meaning |
|---|---|
| `uvicorn` | The command that starts the ASGI web server. FastAPI apps commonly run with uvicorn. |
| `api.serving_api:app` | Load the variable `app` from `api/serving_api.py`. This is the FastAPI application object. |
| `--host 0.0.0.0` | Listen on all network interfaces inside the container, so Docker can forward traffic into it. |
| `--port 8000` | Listen on container port `8000`. |

So `--host` and `--port` are not Docker options here. They are options for the `uvicorn` command. Docker simply runs this command when the container starts.

`EXPOSE 8000` documents the intended container port. It does not publish the port by itself.

Build and run:

```powershell
docker build -f Dockerfile.api -t de-api .
docker run --rm -p 8000:8000 `
  -v ${PWD}/data:/app/data `
  -v ${PWD}/reference/planning-areas.geojson:/app/reference/planning-areas.geojson:ro `
  de-api
```

While the API is running, check **Docker Desktop -> Containers**:

| What to inspect | What it tells you |
|---|---|
| Container status | The API container should be running, because FastAPI is a long-running service. |
| Ports | You should see host port `8000` mapped to container port `8000`. |
| Logs | Uvicorn logs show that the server started and is waiting for requests. |
| Files / mounts if available | The container can see `/app/data` and `/app/reference/planning-areas.geojson`, both mounted from the host. |

Then look at the data effect: the API does not create new pipeline files. It reads the existing `data/basic_ingestion.duckdb` file created by the pipeline container, combines taxi points with the mounted planning-area GeoJSON when needed, and exposes the result as HTTP endpoints.

Open:

- `http://localhost:8000/health`
- `http://localhost:8000/taxi-summary`
- `http://localhost:8000/taxi-latest`
- `http://localhost:8000/docs`

Port mapping:

```text
-p 8000:8000
   |    |
   |    `-- port inside the container
   `------- port on your laptop
```

The API listens on `0.0.0.0:8000` inside the container. Docker forwards traffic from your laptop to that port.


## 8. Containerize the Dashboard

Now write the dashboard image. The dashboard calls the API. It does not read DuckDB directly.

```text
Dashboard container -> API container -> latest taxi points -> static PNG map
```

Your steps:

1. Open `Dockerfile.dashboard`.
2. Make it install `requirements.dashboard.txt`.
3. Copy the `dashboard/` folder into the image.
4. Expose container port `8000`.
5. Start uvicorn on `0.0.0.0:8000`.

Your Dockerfile should look like this:

```dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY requirements.dashboard.txt .
RUN pip install --no-cache-dir -r requirements.dashboard.txt

COPY dashboard ./dashboard

EXPOSE 8000

CMD ["uvicorn", "dashboard.dashboard_app:app", "--host", "0.0.0.0", "--port", "8000"]
```

Keep the API container running. In a second PowerShell window, build and run the dashboard:

```powershell
docker build -f Dockerfile.dashboard -t de-dashboard .
docker run --rm -p 8501:8000 `
  -e TAXI_API_URL=http://host.docker.internal:8000/taxi-latest `
  -e RAINFALL_API_URL=http://host.docker.internal:8000/rainfall-latest `
  -e DATA_DIR=/app/data `
  -e GEOJSON_PATH=/app/reference/planning-areas.geojson `
  -v ${PWD}/data:/app/data `
  -v ${PWD}/reference/planning-areas.geojson:/app/reference/planning-areas.geojson:ro `
  de-dashboard
```

Now check **Docker Desktop -> Containers** again. You should be able to see the running dashboard container, and the API container should still be running in the first terminal.

| Container | What to notice |
|---|---|
| API container | Exposes laptop port `8000`; reads the mounted DuckDB file. |
| Dashboard container | Exposes laptop port `8501`; calls the API through `host.docker.internal:8000` for taxi and rainfall data, then draws a static PNG map. |

This step usually does not modify the DuckDB data. The dashboard is a reader: it asks the API for taxi and rainfall JSON, converts the response into a PNG map, and shows the current contents of the pipeline output.

Open `http://localhost:8501`.

Why these addresses?

| Situation | Address |
|---|---|
| Browser on laptop opens dashboard | `http://localhost:8501` |
| Dashboard container calls API published on laptop | `http://host.docker.internal:8000` for both `TAXI_API_URL` and `RAINFALL_API_URL` |
| Dashboard app listens inside its own container | `0.0.0.0:8000` |

Both API and dashboard can listen on container port `8000` because each container has its own network namespace. The host ports must differ: API uses host `8000`, dashboard uses host `8501`.


## 9. Connect Services with Docker Compose

### 9.1 Why Compose

Manual `docker run` is good for learning one container at a time. It becomes awkward when the system has multiple related services.

Docker Compose moves repeated runtime settings into `docker-compose.yml`:

```text
docker run flags       -> command-line configuration
docker-compose.yml     -> saved multi-service configuration
```

### 9.2 Compose Compared with Build, Run, Exec, and Down

A useful way to place Compose in your mental model:

| Command | Main job | Compose relationship |
|---|---|---|
| `docker build` | Build one image from one Dockerfile. | Compose can do this for each service that has a `build:` section. |
| `docker run` | Start one container with flags such as `-p`, `-v`, and `-e`. | Compose stores those run settings under each service. |
| `docker exec` | Run an extra command inside an already running container. | Compose can target a service with `docker compose exec <service> <command>`. |
| `docker rm` / `docker stop` | Stop or remove individual containers. | `docker compose down` stops and removes the containers and network for the whole Compose project. |

So `docker compose up --build` roughly means: read the YAML file, build images if needed, create the network, then run the services with their saved ports, mounts, environment variables, and dependencies.

Compose is not the same kind of command as `docker exec`. `exec` is for entering or debugging a container that already exists. Compose is for declaring and managing the multi-container application around it.

### 9.3 Read the Services

The provided `docker-compose.yml` defines:

| Service | Dockerfile | Purpose | Port |
|---|---|---|---|
| `pipeline` | `Dockerfile.pipeline` | Fetch API data, clean CSV, update DuckDB | none |
| `api` | `Dockerfile.api` | Serve FastAPI endpoints | host `8000` -> container `8000` |
| `dashboard` | `Dockerfile.dashboard` | Show static PNG taxi/rainfall dashboard | host `8501` -> container `8000` |

### 9.4 Profiles: Optional Service Groups

Notice that `pipeline` has a profile:

```yaml
profiles:
  - tools
```

A Compose profile is an optional service group. Services without a profile start by default when you run `docker compose up`. Services with a profile start only when you explicitly enable that profile.

In this demo, `api` and `dashboard` are always-on services, so they start by default. `pipeline` is different: it is an important project workload, but it is a batch job. It starts, fetches data, writes files and DuckDB tables, then exits. That is why we put it under the `tools` profile and run it only when we want to create or refresh data:

```powershell
docker compose --profile tools run --rm pipeline
```

Read that as: enable the optional `tools` services, run the `pipeline` service once, then remove that one-off container after it exits.

### 9.5 Compose Network and Service Names

When Compose starts services, it creates a private network for this project. Containers in the same Compose project can find each other by **service name**.

That service name works like a small internal DNS name. In this file, the API service is named `api`:

```yaml
services:
  api:
    ports:
      - "8000:8000"
```

So, inside the Compose network, another container can call:

```text
http://api:8000
```

This address is for container-to-container traffic. It is different from the browser address on your laptop:

| Caller | Correct API address |
|---|---|
| Browser on laptop | `http://localhost:8000` |
| Dashboard container in Compose | `http://api:8000` |
| Manually run dashboard container calling host-published API | `http://host.docker.internal:8000` |

That is why the dashboard no longer needs `host.docker.internal` in Compose. Both containers are on the same Compose network, so the dashboard uses the API service name:

```yaml
environment:
  TAXI_API_URL: http://api:8000/taxi-latest
```

`api` is the service name. `8000` is the API container port. `/taxi-latest` is the endpoint that returns the latest taxi coordinates for the dashboard map.

### 9.6 Dependencies and Startup Order

The dashboard also declares that it depends on the API:

```yaml
depends_on:
  - api
```

Read this as: start the `api` service before starting the `dashboard` service.

Important nuance: basic `depends_on` controls startup order, not application readiness. It does not prove the API has finished loading data or is already returning successful responses. For production systems, you usually add health checks and stronger readiness conditions. For this classroom demo, startup order is enough to show the dependency relationship.


### Run the Compose Version

First run the one-off pipeline service:

```powershell
docker compose --profile tools run --rm --build pipeline
```

Then start the API and dashboard together:

```powershell
docker compose up -d --build
docker compose ps
```

Then check **Docker Desktop -> Containers**. Compose groups these containers under one project. You should see `api` and `dashboard` running together, with their published ports visible in the UI.

Open:

- API docs: `http://localhost:8000/docs`
- Dashboard: `http://localhost:8501`

Follow logs:

```powershell
docker compose logs -f
```

Stop and remove the Compose containers and network:

```powershell
docker compose down
```

Common commands:

| Command | Meaning |
|---|---|
| `docker compose build` | Build images defined in the YAML file |
| `docker compose up --build` | Build if needed, then create and start services with logs attached |
| `docker compose up -d --build` | Start services in the background |
| `docker compose ps` | Show containers and port mappings in this Compose project |
| `docker compose logs -f api` | Follow logs for one service |
| `docker compose down` | Remove the Compose containers and network |

Key difference:

| Command style | Best for |
|---|---|
| `docker run ...` | Learning or running one container manually |
| `docker compose ...` | Repeating a multi-container setup with shared network, volumes, env vars, and dependencies |


## 10. Read a Real Compose File: Airflow

After reading our small `docker-compose.yml`, look at a real, production-style Compose file from the Airflow lesson. We will not run Airflow in this Docker lesson; the goal is to practice reading a larger Compose system.

Open this file:

```text
../day_2_2_Airflow/airflow_project/docker-compose.yaml
```

Read it with these questions:

| What to find | Why it matters |
|---|---|
| Services such as `postgres`, `redis`, `airflow-webserver`, `airflow-scheduler`, `airflow-worker`, `airflow-triggerer` | Airflow is a small system, not one process. |
| `ports` | Which services are exposed to your laptop, such as the Airflow UI. |
| `volumes` | Where DAGs, logs, plugins, config, and shared data are mounted. |
| `environment` | How Airflow receives runtime configuration. |
| `depends_on` / health checks | Startup order and readiness matter more in larger systems. |
| `_PIP_ADDITIONAL_REQUIREMENTS` | Convenient for class, but not ideal for production. |
| `AIRFLOW_UID` | Mounted folders and file permissions matter. |

This is the same Compose idea as our smaller demo, just with more services and more operational concerns.

### Why Airflow Becomes Useful

The practice exercise exposes the next problem:

```text
A shell loop can repeat the pipeline, but it is not a real scheduler.
```

If this were a production-like data pipeline, a scheduler such as Airflow would help with:

| Need | Why it matters |
|---|---|
| Schedule | Run the pipeline every minute, hourly, daily, or on a cron expression. |
| Task graph | Represent `fetch -> clean -> load` as separate observable steps. |
| Retries | Retry failed API calls or transient load failures. |
| Logs | Keep task-level logs and history. |
| Status UI | See which task failed and when. |
| Backfills | Re-run missed intervals when appropriate. |

Conceptually, Airflow would replace the simple repeating loop:

```text
Airflow DAG
  -> run taxi/rainfall pipeline tasks on a schedule
  -> write raw JSON, clean CSV, and DuckDB

API container
  -> read DuckDB

Dashboard container
  -> call API and redraw the map
```

That is the bridge to the Airflow lesson. In this Docker lesson, we stop at reading the larger Compose file and understanding why a scheduler is useful.


## 11. Common Problems and Troubleshooting

| Problem | What to check |
|---|---|
| Docker command fails | Is Docker Desktop running? Try `docker run hello-world`. |
| Generated data disappears | Add `-v ${PWD}/data:/app/data`. |
| API cannot find DuckDB | Confirm `data/basic_ingestion.duckdb` exists on the host and the mount path is correct. |
| Browser cannot reach API | Confirm `-p 8000:8000` and server bind `--host 0.0.0.0`. |
| Port already in use | Change the host side, for example `-p 8001:8000`. |
| Dashboard cannot reach API with manual containers | Use `host.docker.internal`, not `localhost`. |
| Dashboard cannot reach API with Compose | Use service name `api`, not `host.docker.internal`. |
| Windows mount fails | Use an absolute host path and confirm Docker Desktop can access the drive. |
| Container exits immediately | The main command finished or failed. Check `docker ps -a` and `docker logs <container_id>`. |


# Docker and Docker Compose Practice

## Practice: Make the Pipeline Repeat

So far, the pipeline runs once and exits. How can you keep refreshing the DuckDB file while the API and dashboard stay running?

**Goal:** add a new Compose service that turns the pipeline image into a lightweight repeating worker. The API and dashboard should keep running, and the dashboard should show newer data after refresh.

**Task:** edit `docker-compose.yml` and add a service named `pipeline-loop`.

Design constraints:

| Requirement | Hint |
|---|---|
| Reuse the same pipeline image | Use the same `build:` settings as `pipeline`. |
| Keep writing to the same host data folder | Use the same `./data:/app/data` volume. |
| Run taxi and rainfall jobs repeatedly | Override the container command. |
| Avoid starting it during normal `docker compose up` | Put it under a profile such as `scheduler`. |
| Make the interval configurable | Use an environment variable such as `PIPELINE_INTERVAL_SECONDS`. |


After adding the service, try running:

```powershell
docker compose --profile scheduler up -d --build pipeline-loop api dashboard
docker compose logs -f pipeline-loop
```

`docker compose logs -f` keeps following the logs. Press `Ctrl+C` when you have seen enough output. This stops log following, but it does **not** stop the `pipeline-loop` container.

Stop only the repeating pipeline loop:

```powershell
docker compose --profile scheduler stop pipeline-loop
docker compose --profile scheduler rm -f pipeline-loop
```

Or stop and remove the whole Compose project, including API and dashboard:

```powershell
docker compose down
```

Then open `http://localhost:8501` and refresh the dashboard after one or two minutes.

**What to observe:**

| Where to look | What should change |
|---|---|
| Docker Desktop | `pipeline-loop`, `api`, and `dashboard` are running in the same Compose project. |
| `docker compose logs -f pipeline-loop` | The taxi and rainfall jobs run repeatedly. |
| Local `data/` folder | The repeating pipeline adds raw JSON, clean CSV, and DuckDB updates; dashboard refreshes add a map PNG. |
| Dashboard | Refreshing the page redraws the static taxi/rainfall map with newer data. |

**Check:** one possible solution is a service like this:

```yaml
pipeline-loop:
  build:
    context: .
    dockerfile: Dockerfile.pipeline
  volumes:
    - ./data:/app/data
    - ./reference/planning-areas.geojson:/app/reference/planning-areas.geojson:ro
  environment:
    TAXI_API_URL: ${TAXI_API_URL:-https://api.data.gov.sg/v1/transport/taxi-availability}
    RAINFALL_API_URL: ${RAINFALL_API_URL:-https://api-open.data.gov.sg/v2/real-time/api/rainfall}
    GEOJSON_PATH: /app/reference/planning-areas.geojson
    PIPELINE_INTERVAL_SECONDS: ${PIPELINE_INTERVAL_SECONDS:-60}
  command:
    - sh
    - -c
    - |
      while true; do
        python -m scripts.taxi_job
        python -m scripts.rainfall_job
        sleep "$${PIPELINE_INTERVAL_SECONDS}"
      done
  profiles:
    - scheduler
```

This keeps writing new pipeline snapshots into the mounted `data/` folder. The API reads the same DuckDB file, and the dashboard uses the API taxi and rainfall endpoints to redraw the map PNG.

**Discussion:** This loop is okay for a Docker practice exercise. It is not a production scheduler. Airflow is a better next step because it gives scheduling, retries, logs, task status, and a UI.

## Thinking Ahead: Airflow and Database Boundaries

### 1. If we add Airflow, what changes?

Think about replacing the `pipeline-loop` container with Airflow:

```text
Airflow DAG
  -> run taxi/rainfall pipeline tasks on a schedule
  -> write raw JSON, clean CSV, and DuckDB

API container
  -> read DuckDB

Dashboard container
  -> call API and redraw the map
```

Airflow would help with things the simple loop does not handle well:

| Need | Why Airflow helps |
|---|---|
| Schedule | Run every minute, hourly, daily, or on a cron expression. |
| Retries | If a network call fails, retry the task instead of silently stopping the loop. |
| Logs | See each task's stdout, error, duration, and history. |
| Task status | Know whether fetch, clean, or load failed. |
| Dependencies | Express `fetch -> clean -> load` clearly. |
| Backfills | Re-run missed historical intervals when appropriate. |

Question: in our classroom setup, should Airflow run the Python functions directly, or should Airflow trigger a pipeline container?

### 2. What are the limits of sharing one DuckDB file?

This demo uses a shared file database:

```text
pipeline/Airflow writes data/basic_ingestion.duckdb
API reads data/basic_ingestion.duckdb
```

That is simple and great for teaching, but it has limits:

| Question | Hint |
|---|---|
| What happens if the API reads while the pipeline writes? | File databases can have concurrency and locking limits. |
| What happens with many API users? | A single local file is not the same as a database server. |
| What happens if the writer partially fails? | The API might see old data, missing tables, or temporary errors unless writes are handled carefully. |
| Can two machines share this file safely? | Local file paths and Docker bind mounts do not automatically become distributed storage. |
| What would production use? | Often PostgreSQL, cloud object storage plus query engine, or another database service with clearer concurrency guarantees. |

For this lesson, DuckDB is a good local analytics database. For a multi-user service, ask whether the API should read from a database server or from published data artifacts instead of a file being actively updated.


## Docker Interview Questions

Use this section after the hands-on work. A good answer should be short, precise, and connected to a concrete example from this notebook.

### 1. What problem does Docker solve?

**Answer:** Docker packages an application with its runtime, dependencies, code, and configuration. It reduces the ?works on my machine? problem because the same image can run on different machines with the same packaged environment.

### 2. What is the difference between an image and a container?

**Answer:** An image is a reusable package or template. A container is a running or stopped instance created from an image.

### 3. What does `FROM python:3.11-slim` mean in a Dockerfile?

**Answer:** It uses an existing Python image as the base image. The new image adds project-specific files, dependencies, and startup commands on top of that base.

### 4. What is the final `.` in `docker build -f Dockerfile.pipeline -t de-pipeline .`?

**Answer:** The final `.` is the build context. It means Docker can use files from the current folder during the build. `COPY` commands can only copy files from inside this context.

### 5. Why copy `requirements.txt` before copying source code?

**Answer:** Docker builds images in layers. If the dependency file does not change, Docker can reuse the package installation layer even when the application code changes.

### 6. Why did generated files disappear after running the first pipeline container?

**Answer:** The files were written inside the container filesystem. When the container was removed with `--rm`, those files disappeared. A bind mount such as `-v ${PWD}/data:/app/data` keeps output on the host machine.

### 7. What is the difference between a bind mount and port mapping?

**Answer:** A bind mount shares files between the host and container. Port mapping forwards network traffic from a host port to a container port.

### 8. What does `docker exec -it` do?

**Answer:** It runs an interactive command inside an already running container. For example, `docker exec -it pipeline-inspect sh` opens a shell so you can inspect paths, files, and environment from inside the container.

### 9. Why does a web server inside a container usually listen on `0.0.0.0`?

**Answer:** `0.0.0.0` means the server listens on all network interfaces inside the container. This allows Docker port mapping to forward traffic from the host into the container.

### 10. Why can `localhost` be wrong inside Docker?

**Answer:** `localhost` means ?myself.? In a container, `localhost` refers to that container, not the laptop and not another container.

### 11. When should you use `host.docker.internal`?

**Answer:** Use it when a container needs to call a service running directly on the host machine, such as a local FastAPI server on the laptop.

### 12. Why can the dashboard call `http://api:8000` in Docker Compose?

**Answer:** Docker Compose creates a private network for the services. Service names become DNS names, so `api` resolves to the API container.

### 13. What is the difference between `docker run` and Docker Compose?

**Answer:** `docker run` starts one container manually. Docker Compose defines and runs multiple related services with saved configuration for ports, volumes, environment variables, dependencies, and networks.

### 14. How can you share an image without pushing it to Docker Hub?

**Answer:** Use `docker save` to export the image as a tar file and `docker load` to load it on another machine.

```powershell
docker save -o de-dashboard-day2.tar de-dashboard:day2
docker load -i de-dashboard-day2.tar
```

Use `docker save/load` for images. `docker export/import` works with container filesystems and may lose image metadata such as the default command.

### 15. What should you check if a dashboard container cannot reach an API container?

**Answer:** Check where the request comes from. With manual containers, use `host.docker.internal` if the API is published on the laptop. With Compose, use the service name such as `http://api:8000`. Also check port mappings, logs, and whether the API container is running.


## Wrap-up

Today you built the architecture incrementally:

```text
hello-world
    |
    v
pipeline image
    |
    v
bind-mounted data folder
    |
    v
API image -- host 8000 -> container 8000
    |
    v
dashboard image -- host 8501 -> container 8000
    |
    v
Docker Compose orchestration

```

Final takeaway:

> Build one responsibility at a time. Share files with mounts, share network access with ports, and coordinate related services with Docker Compose.
